在收集前須更改以下設定
1. SUB_CATEGORIES = {
    #'出版': '4',
    #'時尚': '7',
    #'設計': '8',
    #'科技': '11',
    #'教育': '12',
    '遊戲': '13',
    #'飲食': '14',
    #'社會': '15'
}
選擇要爬蟲的類別

2. BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/XX_New_ZecZec_Dataset'
將XX換成要爬蟲的類別

EX:如果現在要爬出版，那設定就是
SUB_CATEGORIES = {
    '出版': '4',
    #'時尚': '7',
    #'設計': '8',
    #'科技': '11',
    #'教育': '12',
    #'遊戲': '13',
    #'飲食': '14',
    #'社會': '15'
}
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/出版_New_ZecZec_Dataset'

※ 收集的資料量變多，建議多準備幾隻google帳號

In [ ]:
# 1. 清理Colab預設可能衝突的舊版套件
!apt-get remove chromium-browser chromium-chromedriver -y
!apt-get autoremove -y

# 2. 下載並安裝官方Google Chrome穩定版
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

# 3. 安裝Linux虛擬螢幕套件
!apt-get install -y xvfb
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt-get install -y nodejs
!apt-get install -y ffmpeg

# 4. 一次裝齊所有需要的Python爬蟲套件
!pip install undetected-chromedriver pyvirtualdisplay selenium beautifulsoup4 requests pandas
!pip install -U --pre "yt-dlp[default]"

!curl -fsSL https://deno.land/x/install/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
--2026-07-22 14:59:25--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 142.250.125.136, 142.250.125.190, 142.250.125.91, ...
Connecting to dl.google.com (dl.google.com)|142.250.125.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 133533368 (127M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 127.35M   387MB/s    in 0.3s    

2026-07-22 14:59:25 (387 MB/s) - ‘google-ch

In [ ]:
#連線測試
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
import time

# 1. 啟動虛擬螢幕
print("正在架設虛擬顯示器...")
display = Display(visible=0, size=(1920, 1080))
display.start()

# 2. 設定啟動參數
options = uc.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

options.binary_location = "/usr/bin/google-chrome"

print("啟動Chrome...")
driver = uc.Chrome(options=options, version_main=148)

print("嘗試連線至嘖嘖...")
driver.get("https://www.zeczec.com/")

# 給Cloudflare更充裕的時間跑完JavaScript
time.sleep(8)

print("目前網頁標題：", driver.title)

# 測試完關閉資源
driver.quit()
display.stop()

正在架設虛擬顯示器...
啟動Chrome...


FileNotFoundError: 
---------------------
Could not determine browser executable.
---------------------
Make sure your browser is installed in the default location (path).
If you are sure about the browser executable, you can specify it using
the `browser_executable_path='/path/to/browser/executable` parameter.



In [ ]:
#爬蟲跑到一半當機，或手動按了停止鈕後，用來釋放RAM的
!pkill -f chrome

In [ ]:
import os
import re
import time
import csv
import requests
import pandas as pd
import yt_dlp
import subprocess
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from pyvirtualdisplay import Display
from selenium.webdriver.common.by import By
from google.colab import drive
import shutil

if os.path.exists('/root/.deno/bin'):
    os.environ['PATH'] = '/root/.deno/bin:' + os.environ.get('PATH', '')

# ==========================================
# 新增功能區：1. 專案自動編號  2. FAQ 收集器  3. 專案日期抓取
# ==========================================
# 1.自動編號
category_prefix_map = {
    "出版": "P", "時尚": "F", "設計": "D", "科技": "T",
    "教育": "E", "遊戲": "G", "飲食": "FD", "社會": "S"
}
status_prefix_map = {"成功": "S", "失敗": "F"}
project_counters = {}

def generate_project_code(category_name, status):
    """產生如 GF1 (遊戲+失敗+第1號) 的編號"""
    cat_p = category_prefix_map.get(category_name, "X")
    stat_p = status_prefix_map.get(status, "U")
    prefix = f"{cat_p}{stat_p}"

    if prefix not in project_counters:
        project_counters[prefix] = 1
    else:
        project_counters[prefix] += 1
    return f"{prefix}{project_counters[prefix]}"

# ==========================================
# 1.5 斷點續傳掃描器 (掃描雲端硬碟已完成的專案)
# ==========================================
def load_resume_state(base_dir):
    """掃描已建立的資料夾，還原爬蟲進度與專案編號"""
    scraped_slugs = set()
    category_counts = {}

    global project_counters
    project_counters = {} # 初始化並重新計算編號

    if not os.path.exists(base_dir):
        return scraped_slugs, category_counts

    for main_name in MAIN_TYPES.keys():
        for sub_name in SUB_CATEGORIES.keys():
            for status in ['成功', '失敗']:
                status_dir = os.path.join(base_dir, main_name, sub_name, status)
                if os.path.exists(status_dir):
                    # 抓出該狀態下的所有專案資料夾
                    folders = [f for f in os.listdir(status_dir) if os.path.isdir(os.path.join(status_dir, f)) and "_" in f]
                    category_counts[(main_name, sub_name, status)] = len(folders)

                    for f in folders:
                        # 解析檔名，例如 GF1_rogerems
                        try:
                            proj_code, slug = f.split("_", 1)
                            scraped_slugs.add(slug)

                            # 讓自動編號 (GF1, GS2) 接續目前的數字
                            match = re.match(r'([A-Z]+)(\d+)', proj_code)
                            if match:
                                prefix, num = match.group(1), int(match.group(2))
                                if prefix not in project_counters or num > project_counters[prefix]:
                                    project_counters[prefix] = num
                        except:
                            pass

    return scraped_slugs, category_counts

# 2.FAQ收集
def process_faq_data(driver, project_url, proj_code, save_folder_path, funding_days=30):
    """抓取 FAQ 並回傳 (總數, 更新頻率)"""
    try:
        driver.get(f"{project_url}/faqs")
        time.sleep(3) # 【關鍵】必須等待標籤切換與動態框框渲染完畢

        total_count = 0
        tabs = driver.find_elements(By.XPATH, "//*[contains(text(), '常見問答')]")
        for tab in tabs:
            match = re.search(r'常見問答\s*(\d+)', tab.text)
            if match:
                total_count = int(match.group(1))
                break

        update_freq = round(total_count / funding_days, 4) if funding_days > 0 else 0

        if total_count > 0:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            faq_text_content = f"專案編號：{proj_code}\nFAQ 總體數：{total_count}\n更新頻率：{update_freq}\n" + "="*30 + "\n\n"

            date_texts = soup.find_all(string=re.compile(r'更新於\s*\d{4}'))

            if date_texts:
                for i, date_text in enumerate(date_texts, 1):
                    current_node = date_text.parent
                    while current_node.parent and len(current_node.parent.find_all(string=re.compile(r'更新於\s*\d{4}'))) == 1:
                        current_node = current_node.parent

                    if current_node:
                        text = current_node.get_text(separator='\n', strip=True)
                        faq_text_content += f"【第 {i} 題】\n{text}\n\n"

            else:
                faq_text_content += soup.get_text(separator='\n', strip=True)[:2000]

            txt_file_path = os.path.join(save_folder_path, f"{proj_code}_FAQ.txt")
            with open(txt_file_path, "w", encoding="utf-8") as f:
                f.write(faq_text_content)

        return total_count, update_freq
    except Exception as e:
        print(f"FAQ抓取失敗: {e}")
        return 0, 0

# === 3. 專案日期抓取 ===
def get_project_dates(page_source):
    """抓取專案的開始與結束日期"""
    start_date_str = ""
    end_date_str = ""
    try:
        soup = BeautifulSoup(page_source, 'html.parser')
        date_text_div = soup.find(string=re.compile(r'\d{4}/\d{2}/\d{2}.+–.+\d{4}/\d{2}/\d{2}'))

        if date_text_div:
            dates = re.findall(r'\d{4}/\d{2}/\d{2}', date_text_div)
            if len(dates) >= 2:
                start_date_str = dates[0].replace('/', '-')
                end_date_str = dates[1].replace('/', '-')
    except Exception as e:
        print(f"日期抓取失敗: {e}")

    return start_date_str, end_date_str

# ==========================================
# 1. 核心功能：多媒體下載器
# ==========================================
def zeczec_multimodal_downloader(url, project_name, driver, base_folder_path, proj_code, valid_img_urls):
    img_path = os.path.join(base_folder_path, f"{proj_code}_images")
    video_path = os.path.join(base_folder_path, f"{proj_code}_videos")
    os.makedirs(img_path, exist_ok=True)
    os.makedirs(video_path, exist_ok=True)

    print(f" [影音任務] 正在掃描：{project_name}")

    driver.get(url)
    time.sleep(3)

    for i in range(3):
        driver.execute_script(f"window.scrollTo(0, {i * 1200});")
        time.sleep(1.2)

    resource_log = []

    # --- [A. 圖片抓取] ---
    print("正在掃描圖片...")
    print(f"準備下載 {len(valid_img_urls)} 張內容圖片...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    for idx, img_url in enumerate(valid_img_urls):
        try:
            resp = requests.get(img_url, headers=headers, timeout=10)
            if resp.status_code == 200:
                ctype = resp.headers.get('Content-Type', '').lower()
                ext = ".gif" if 'gif' in ctype else ".png" if 'png' in ctype else ".webp" if 'webp' in ctype else ".jpg"

                f_name = f"img_{idx}{ext}"
                f_path = os.path.join(img_path, f_name)

                with open(f_path, 'wb') as f:
                    f.write(resp.content)

                if os.path.getsize(f_path) > 5000:
                    resource_log.append({"FileName": f_name, "Type": "Image", "URL": img_url})
                else:
                    os.remove(f_path)
        except Exception as e:
            print(f"圖片 img_{idx} 下載失敗: {e}")
            continue

    # --- [B. 影片偵測] ---
    video_urls = set()
    try:
        for iframe in driver.find_elements(By.TAG_NAME, "iframe"):
            v_src = iframe.get_attribute("src") or iframe.get_attribute("data-src")
            if v_src and "youtube.com" in v_src:
                video_urls.add(v_src.split('?')[0].replace("embed/", "watch?v="))
    except: pass

    try:
        for ns in driver.find_elements(By.TAG_NAME, "noscript"):
            ns_content = ns.get_attribute("innerHTML")
            if "youtube.com" in ns_content and "embed" in ns_content:
                match = re.search(r'embed/([a-zA-Z0-9_-]{11})', ns_content)
                if match:
                    video_urls.add(f"https://www.youtube.com/watch?v={match.group(1)}")
    except: pass

    # --- [C. 影片下載] ---
    cookie_file_path = '/content/drive/MyDrive/ZecZec_Group_Data/www.youtube.com_cookies.txt'

    if not os.path.exists(cookie_file_path):
      print(f"⚠️ 找不到 cookie 檔案：{cookie_file_path}，將以未登入狀態嘗試下載")
      cookie_file_path = None

    if video_urls:
        ydl_opts = {
              'format': 'bestvideo+bestaudio/best',
              'outtmpl': f'{video_path}/video_%(title).30s_%(id)s.%(ext)s',
              'windowsfilenames': True,
              'quiet': True,
              'no_warnings': True,
              'ignoreerrors': True,
              'sleep_interval': 8,
              'max_sleep_interval': 20,
              'cookiefile': cookie_file_path,
              'js_runtimes': {'deno': {}},
              'extractor_args': {
                  'youtube': {'player_client': ['android', 'mweb', 'web_safari', 'ios']}
              },
              'retries': 3
          }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            for v_url in video_urls:
                try:
                    print(f"下載影片: {v_url}")
                    ydl.download([v_url])
                    resource_log.append({"FileName": v_url, "Type": "Video", "URL": v_url})
                except Exception as e: print(f"影片下載失敗，可能原因:{e}")

    if resource_log:
        pd.DataFrame(resource_log).to_csv(f"{base_folder_path}/{proj_code}_resource_log.csv", index=False, encoding="utf-8-sig")

# ==========================================
# 2. 環境與初始化設定
# ==========================================
MAIN_TYPES = {'群眾募資': '0', '預購式專案': '1'}
SUB_CATEGORIES = {
    #'出版': '4',
    #'時尚': '7',
    #'設計': '8',
    #'科技': '11',
    #'教育': '12',
    '遊戲': '13',
    #'飲食': '14',
    #'社會': '15'
}

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data/遊戲_New_ZecZec_Dataset'

def create_directories():
    if not os.path.exists(BASE_DIR): os.mkdir(BASE_DIR)
    for main_name in MAIN_TYPES.keys():
        main_path = os.path.join(BASE_DIR, main_name)
        if not os.path.exists(main_path): os.mkdir(main_path)
        for sub_name in SUB_CATEGORIES.keys():
            sub_path = os.path.join(main_path, sub_name)
            if not os.path.exists(sub_path): os.mkdir(sub_path)
            for status_name in ['成功', '失敗']:
                status_path = os.path.join(sub_path, status_name)
                if not os.path.exists(status_path): os.mkdir(status_path)
    print("資料夾結構建立完成！")

def ensure_chrome_installed():
    if not os.path.exists("/usr/bin/google-chrome"):
        print("⚠️ 偵測不到 Chrome，重新安裝中...")
        subprocess.run("apt-get remove -y chromium-browser chromium-chromedriver", shell=True)
        subprocess.run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb", shell=True)
        subprocess.run("apt install -y /tmp/chrome.deb", shell=True)
        subprocess.run("apt-get -f install -y", shell=True)
        if not os.path.exists("/usr/bin/google-chrome"):
            raise RuntimeError("Chrome 安裝失敗，請檢查 apt 是否有錯誤訊息。")
    print("✅ Chrome 已就緒")

def get_chrome_major_version():
    out = subprocess.check_output(["/usr/bin/google-chrome", "--version"]).decode()
    match = re.search(r'(\d+)\.\d+\.\d+\.\d+', out)
    if not match:
        raise RuntimeError(f"無法解析 Chrome 版本號，原始輸出：{out}")
    return int(match.group(1))

def get_driver():
    ensure_chrome_installed()
    major_version = get_chrome_major_version()
    print(f"啟動Chrome...(偵測到版本 {major_version})")

    display = Display(visible=0, size=(1920, 1080))
    display.start()

    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.binary_location = "/usr/bin/google-chrome"

    driver = uc.Chrome(options=options, version_main=major_version)
    return driver, display

# ==========================================
# 3. 核心解析邏輯 (文字抓取)
# ==========================================
def scrape_project_details(driver, project_url, main_name, sub_name):
    driver.get(project_url)
    time.sleep(3)

    try:
        xpath_query = "//*[self::button or self::a or self::input][contains(., '我已滿 18 歲')]"
        age_btn = driver.find_element(By.XPATH, xpath_query)
        driver.execute_script("arguments[0].click();", age_btn)
        time.sleep(4)
    except: pass

    try:
        expand_btn = driver.find_element(By.CSS_SELECTOR, 'button.js-expand-project')
        driver.execute_script("arguments[0].click();", expand_btn)
    except: pass

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    project_id = project_url.split('/')[-1].split('?')[0]

    og_title = soup.find('meta', property='og:title')
    if og_title and og_title.get('content'):
        project_title = og_title['content']
    elif soup.title:
        project_title = soup.title.get_text(strip=True)
    else:
        project_title = project_id

    project_title = project_title.replace("嘖嘖 | ", "").replace(" | 嘖嘖", "").replace("zeczec", "").strip()

    pct_tag = soup.find(class_='js-percentage-raised')
    if not pct_tag: return "ERROR"

    time_left_tag = soup.find(class_='js-time-left')
    if time_left_tag and any(x in time_left_tag.text for x in ['天', '時', '分']):
        return "ACTIVE"

    pct_val = int(re.sub(r'[^\d]', '', pct_tag.text)) if re.sub(r'[^\d]', '', pct_tag.text).isdigit() else 0
    status_name = '成功' if pct_val >= 100 else '失敗'

    content_div = soup.find('div', id='project_content')
    target_soup = content_div if content_div else soup

    valid_img_urls = []
    img_index = 0

    for img in target_soup.find_all('img'):
        img_url = img.get('data-src') or img.get('src')
        if not img_url: continue

        if img_url.startswith("data:") or any(x in img_url.lower() for x in ['icon', 'logo', 'avatar', '.svg']):
            continue

        valid_img_urls.append(img_url)
        img.insert_after(f"\n[img_{img_index}]\n")
        img_index += 1

    content_text = target_soup.get_text(separator='\n', strip=True) if content_div else "無法抓取內文"

    price_tags = soup.find_all('div', class_='text-black font-bold text-xl flex items-center')
    plan_prices = []
    plans_details = []

    for idx, pt in enumerate(price_tags, 1):
        price_match = re.search(r'NT\$\s*([\d,]+)', pt.text)
        if price_match:
            plan_prices.append(price_match.group(1).replace(',', ''))

        card = pt.parent.parent
        if len(card.get_text(strip=True)) < 30:
            card = card.parent

        if card:
            for img in card.find_all('img'):
                img_url = img.get('data-src') or img.get('src')
                if not img_url: continue

                if img_url.startswith("data:") or any(x in img_url.lower() for x in ['icon', 'logo', 'avatar', '.svg']):
                    continue

                valid_img_urls.append(img_url)
                img.insert_after(f"\n[img_{img_index}]\n")
                img_index += 1

            card_text = card.get_text(separator='\n', strip=True)
            plans_details.append(f"▼ 【方案 {idx}】 ▼\n{card_text}")

    prices_str = " | ".join(plan_prices) if plan_prices else "無價格資訊"
    plans_text_output = "\n\n----------------------------------------\n\n".join(plans_details) if plans_details else "無方案資訊"

    actual_days = 30
    try:
        date_text_div = soup.find(string=re.compile(r'\d{4}/\d{2}/\d{2}.+–.+\d{4}/\d{2}/\d{2}'))
        if date_text_div:
            dates = re.findall(r'\d{4}/\d{2}/\d{2} \d{2}:\d{2}', date_text_div)
            if len(dates) == 2:
                from datetime import datetime
                start_date_dt = datetime.strptime(dates[0], "%Y/%m/%d %H:%M")
                end_date_dt = datetime.strptime(dates[1], "%Y/%m/%d %H:%M")
                delta_days = (end_date_dt - start_date_dt).days
                actual_days = max(1, delta_days)
    except Exception as e:
        print(f"天數計算失敗，使用預設值 30 天: {e}")
        actual_days = 30

    start_date_str, end_date_str = get_project_dates(driver.page_source)

    return {
        "status": status_name,
        "project_id": project_id,
        "project_title": project_title,
        "content_text": content_text,
        "plans_text": plans_text_output,
        "valid_img_urls": valid_img_urls,
        "actual_days": actual_days,
        "pct_val": pct_val,
        "start_date": start_date_str,
        "end_date": end_date_str,
        "csv_row": [main_name, sub_name, status_name, project_id, prices_str, len(plan_prices)]
    }

# ==========================================
# 5. 主爬蟲流程 (依指示：邊界觸發式年份溯源與中斷點接續)
# ==========================================
def main_crawler():
    create_directories()

    # 讀取雲端硬碟先前的進度，用以防止收集重複專案
    scraped_slugs, category_counts = load_resume_state(BASE_DIR)
    print(f"\n已讀取中斷點：發現 {len(scraped_slugs)} 個已完成的專案，在新的一輪進行時將自動跳過。")

    driver, display = get_driver()
    csv_filename = os.path.join(BASE_DIR, 'projects_summary.csv')

    if not os.path.exists(csv_filename):
        with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數', 'FAQ總題數', 'FAQ更新頻率', '開始日期', '結束日期', '總天數', '達標率(%)'])

    for main_name, main_type in MAIN_TYPES.items():
        for sub_name, sub_cat in SUB_CATEGORIES.items():
            window_idx = 0
            current_start_date = f"{2025 - window_idx}-07-01"
            current_end_date = f"{2026 - window_idx}-06-30"

            curr_success = category_counts.get((main_name, sub_name, '成功'), 0)
            curr_failed = category_counts.get((main_name, sub_name, '失敗'), 0)

            print(f"\n=======================================================")
            print(f" ⏳ [啟動分類: {sub_name}] 初始時間窗：{current_start_date} ~ {current_end_date}")
            print(f"    ↳ 目前已累積進度：成功 {curr_success} 筆, 失敗 {curr_failed} 筆")
            print(f"=======================================================")

            page_num = 1
            category_done = False

            # 不在分頁重置，一頁頁依序掃描到底
            while not category_done:
                print(f"  [掃描分頁] {sub_name} 第 {page_num} 頁 (目前時間窗: {current_start_date}~{current_end_date} | 累積成功: {curr_success}, 失敗: {curr_failed})...")
                list_url = f"https://www.zeczec.com/categories?category={sub_cat}&type={main_type}&page={page_num}"

                try:
                    driver.get(list_url)
                except:
                    print("瀏覽器連線中斷，正在重啟...")
                    driver.quit(); display.stop()
                    driver, display = get_driver()
                    continue

                time.sleep(3)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                all_cards = soup.select('a[href^="/projects/"]')
                project_urls = []
                for card in all_cards:
                    href = card.get('href')
                    if 'comments' in href or 'updates' in href: continue

                    card_text = card.get_text(strip=True)
                    if any(keyword in card_text for keyword in ['專案問卷調查中', '即將啟動', '敬請期待', '專案準備中', '即將開始']):
                        print(f"  [略過預熱專案]: {href.split('/')[-1]}")
                        continue

                    project_urls.append(href)
                project_urls = list(dict.fromkeys(project_urls))

                if not project_urls:
                    print(f"\n  🏁 [{sub_name}] 已無更多分頁，歷史資料掃描完畢！最終：成功 {curr_success} 筆, 失敗 {curr_failed} 筆。")
                    break

                for url in project_urls:
                    project_slug = url.split('/')[-1].split('?')[0]

                    # 【需求 3】防止收集重複專案
                    if project_slug in scraped_slugs:
                        print(f"  [重複專案，自動跳過]: {project_slug}")
                        continue

                    full_url = "https://www.zeczec.com" + url

                    try:
                        result = scrape_project_details(driver, full_url, main_name, sub_name)
                    except Exception as e:
                        print(f"專案解析失敗，嘗試重啟瀏覽器: {e}")
                        driver.quit(); display.stop()
                        driver, display = get_driver()
                        continue

                    if result in ["ACTIVE", "ERROR"]: continue

                    proj_start_date = result.get("start_date", "")
                    proj_end_date = result.get("end_date", "")
                    status = result["status"]

                    if not proj_end_date:
                        continue

                    # 💡【核心修正與判定機制】
                    # 當開始爬到結束日期小於目前期間的起始日 (proj_end_date < current_start_date)，
                    # 代表「目前這個一年的資料期間已經完整收完」！
                    while proj_end_date < current_start_date:
                        print(f"\n  📅 偵測到專案結束日 ({proj_end_date}) 早於目前起始日 ({current_start_date})！")
                        print(f"  ↳ 代表目前期間 ({current_start_date} ~ {current_end_date}) 之資料已收齊。")
                        print(f"  ↳ 檢驗累積進度：成功 {curr_success} 筆, 失敗 {curr_failed} 筆 (目標門檻：成功與失敗皆 >= 50 筆)")

                        # 檢查是否收集 >= 50 筆資料 (直到失敗和成功的募資案都收集到50筆(含)以上)
                        if curr_success >= 50 and curr_failed >= 50:
                            print(f"  🎉 成功與失敗皆已達到 >= 50 筆！本分類 ({sub_name}) 蒐集達標，順利結束！")
                            category_done = True
                            break
                        else:
                            # 若為否，就進入下一個一年的資料期間，從上次的中斷點繼續循環
                            window_idx += 1
                            current_start_date = f"{2025 - window_idx}-07-01"
                            current_end_date = f"{2026 - window_idx}-06-30"
                            print(f"  🔄 尚未完全達標！立即進入下一個一年的資料期間 ({current_start_date} ~ {current_end_date})，從中斷點接續繼續...")

                            if window_idx >= 10:
                                print(f"  🛑 [{sub_name}] 已往前溯源超過 10 年，歷史資料已耗盡，停止本分類。")
                                category_done = True
                                break

                    if category_done:
                        break

                    # 如果結束日期大於目前的結束日 (例如剛跨入下一年窗口時，遇到少數跨期較長的專案)，直接跳過
                    if proj_end_date > current_end_date:
                        print(f"  [非目前期間，跳過]: {project_slug} (結束日: {proj_end_date} | 目前期間: {current_start_date} ~ {current_end_date})")
                        continue

                    # 符合目前時間窗口，進行檔案儲存與下載
                    proj_code = generate_project_code(sub_name, status)
                    full_folder_name = f"{proj_code}_{project_slug}"

                    project_folder = os.path.join(BASE_DIR, main_name, sub_name, status, full_folder_name)
                    os.makedirs(project_folder, exist_ok=True)

                    content_filename = f"{proj_code}_content.txt"
                    content_filepath = os.path.join(project_folder, content_filename)

                    with open(content_filepath, 'w', encoding='utf-8') as f:
                        f.write(f"專案名稱：{result['project_title']}\n")
                        f.write("=" * 50 + "\n\n")
                        f.write(result["content_text"])

                    plans_filename = f"{proj_code}_plans.txt"
                    plans_filepath = os.path.join(project_folder, plans_filename)

                    with open(plans_filepath, 'w', encoding='utf-8') as f:
                        f.write(f"專案名稱：{result['project_title']} (方案列表)\n")
                        f.write("=" * 50 + "\n\n")
                        f.write(result["plans_text"])

                    funding_duration = result.get("actual_days", 30)
                    faq_count, faq_freq = process_faq_data(driver, full_url, proj_code, project_folder, funding_days=funding_duration)
                    print(f"  ↳ [進度] 總天數: {funding_duration} 天 | 問答總數: {faq_count} 題 | 計算出頻率: {faq_freq} | 期間: {proj_start_date} ~ {proj_end_date}")

                    updated_csv_row = result["csv_row"]
                    updated_csv_row[3] = proj_code
                    updated_csv_row.extend([faq_count, faq_freq, proj_start_date, proj_end_date, funding_duration, result["pct_val"]])

                    with open(csv_filename, 'a', newline='', encoding='utf-8') as csvfile:
                        writer = csv.writer(csvfile)
                        writer.writerow(updated_csv_row)

                    zeczec_multimodal_downloader(full_url, project_slug, driver, project_folder, proj_code, result["valid_img_urls"])

                    scraped_slugs.add(project_slug)

                    if status == '成功':
                        curr_success += 1
                        category_counts[(main_name, sub_name, '成功')] = curr_success
                    else:
                        curr_failed += 1
                        category_counts[(main_name, sub_name, '失敗')] = curr_failed

                    print(f"完成：{project_slug} ({status}) | 目前累積 -> 成功: {curr_success}, 失敗: {curr_failed}")

                page_num += 1

    driver.quit()
    display.stop()
    print("\n所有採集任務已順利結束！")

# 執行
if __name__ == "__main__":
    main_crawler()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
資料夾結構建立完成！

已讀取中斷點：發現 65 個已完成的專案，在新的一輪進行時將自動跳過。
✅ Chrome 已就緒
啟動Chrome...(偵測到版本 150)

 ⏳ [啟動分類: 遊戲] 初始時間窗：2025-07-01 ~ 2026-06-30
    ↳ 目前已累積進度：成功 17 筆, 失敗 16 筆
  [掃描分頁] 遊戲 第 1 頁 (目前時間窗: 2025-07-01~2026-06-30 | 累積成功: 17, 失敗: 16)...
  [重複專案，自動跳過]: Hertha-LazyLAMB
  [重複專案，自動跳過]: dissecting-the-cold-white
  [重複專案，自動跳過]: accusation-a-cardgame
  [重複專案，自動跳過]: Hero-Mission
  [重複專案，自動跳過]: WoodbudStoryHousePuzzlePack01
  [重複專案，自動跳過]: faustalptraum10anniversary
  [掃描分頁] 遊戲 第 2 頁 (目前時間窗: 2025-07-01~2026-06-30 | 累積成功: 17, 失敗: 16)...
  [重複專案，自動跳過]: ekholux-vr
  [重複專案，自動跳過]: thebouncingnotes
  [重複專案，自動跳過]: escapewelt-thelostsunstone
  [重複專案，自動跳過]: DREAMBASEBALL5
  [重複專案，自動跳過]: Puerto_Rico
  [重複專案，自動跳過]: snoopywoodenpuzzle
  [重複專案，自動跳過]: seti
  [重複專案，自動跳過]: jias-party
  [重複專案，自動跳過]: hoshiorigakuen
  [重複專案，自動跳過]: idventure-Wonderbox-of-Alice
  [重複專案，自動跳過]: lami-a-cup-of-code-

ERROR: [youtube] Pte-wtJMWpk: Join this channel from your computer or Android app to get access to members-only content like this video.


完成：vgc2024 (成功) | 目前累積 -> 成功: 30, 失敗: 16
  [掃描分頁] 遊戲 第 4 頁 (目前時間窗: 2024-07-01~2025-06-30 | 累積成功: 30, 失敗: 16)...
  ↳ [進度] 總天數: 46 天 | 問答總數: 2 題 | 計算出頻率: 0.0435 | 期間: 2024-07-20 ~ 2024-09-05
 [影音任務] 正在掃描：pagui2024
正在掃描圖片...
準備下載 52 張內容圖片...
下載影片: https://www.youtube.com/watch?v=_ROwWZVpbaA
完成：pagui2024 (成功) | 目前累積 -> 成功: 31, 失敗: 16
  ↳ [進度] 總天數: 35 天 | 問答總數: 5 題 | 計算出頻率: 0.1429 | 期間: 2024-07-16 ~ 2024-08-20
 [影音任務] 正在掃描：MyBestFriend
正在掃描圖片...
準備下載 27 張內容圖片...
下載影片: https://www.youtube.com/watch?v=v5WLUKawy30
完成：MyBestFriend (成功) | 目前累積 -> 成功: 32, 失敗: 16
  ↳ [進度] 總天數: 77 天 | 問答總數: 6 題 | 計算出頻率: 0.0779 | 期間: 2024-07-10 ~ 2024-09-26
 [影音任務] 正在掃描：idventure-cluepuzzle
正在掃描圖片...
準備下載 77 張內容圖片...
下載影片: https://www.youtube.com/watch?v=UJonLyNhrrU
完成：idventure-cluepuzzle (成功) | 目前累積 -> 成功: 33, 失敗: 16

  📅 偵測到專案結束日 (2024-06-20) 早於目前起始日 (2024-07-01)！
  ↳ 代表目前期間 (2024-07-01 ~ 2025-06-30) 之資料已收齊。
  ↳ 檢驗累積進度：成功 33 筆, 失敗 16 筆 (目標門檻：成功與失敗皆 >= 50 筆)
  🔄 尚未完全達標！立即進入下一個一年的資料期間 (2023-07-01 ~ 2024-06-30)，從中斷點

ERROR: unable to download video data: HTTP Error 403: Forbidden


下載影片: https://www.youtube.com/watch?v=UI0v12W3sg0


...

ERROR: unable to download video data: HTTP Error 403: Forbidden


下載影片: https://www.youtube.com/watch?v=ihdWr0P9uac
下載影片: https://www.youtube.com/watch?v=85CKRgm97uE


ERROR: unable to download video data: HTTP Error 403: Forbidden


下載影片: https://www.youtube.com/watch?v=ZNGAMDRlkmk
完成：jundoogame (成功) | 目前累積 -> 成功: 45, 失敗: 16
  ↳ [進度] 總天數: 100 天 | 問答總數: 7 題 | 計算出頻率: 0.07 | 期間: 2023-12-08 ~ 2024-03-17
 [影音任務] 正在掃描：re-burn
正在掃描圖片...
準備下載 33 張內容圖片...
下載影片: https://www.youtube.com/watch?v=R_6gVu1hN84
下載影片: https://www.youtube.com/watch?v=1CUtW_sd3UY
下載影片: https://www.youtube.com/watch?v=FJg638Nlm_Y
完成：re-burn (成功) | 目前累積 -> 成功: 46, 失敗: 16
  ↳ [進度] 總天數: 40 天 | 問答總數: 3 題 | 計算出頻率: 0.075 | 期間: 2023-11-28 ~ 2024-01-08
 [影音任務] 正在掃描：tentacles_party_with_nuns
正在掃描圖片...
準備下載 81 張內容圖片...
下載影片: https://www.youtube.com/watch?v=3-MZW3bPQFk
下載影片: https://www.youtube.com/watch?v=83vpSvvsMsc
下載影片: https://www.youtube.com/watch?v=oRzqUhcupOw
下載影片: https://www.youtube.com/watch?v=MHSrrQDozkc
下載影片: https://www.youtube.com/watch?v=MPLJZDkATQU


KeyboardInterrupt: 